# **Project: Cambodian Tourism Chatbot — Prediction & Error Analysis**
---


### **1. Prediction Overview**

> In this notebook, we load the trained **SimpleRNN** model from `train.ipynb` and use it to generate answers for new user questions. We also test the chatbot on different categories of inputs (in-domain, paraphrase, long, short, out-of-vocabulary, off-topic) to find out **where and why the chatbot fails**.
---


### **2. Environment Setup**

Import the libraries we need to load the trained model and run predictions.


In [2]:
# Import libraries
import numpy as np
import pandas as pd
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

### **3. Load the Saved Model and Tokenizer**
#### **3.1 Load Artifacts**
**Instructions**: Load the `model.h5`, `tokenizer.pkl`, and `config.pkl` files that were saved at the end of `train.ipynb`.


In [3]:
# Load the trained model
model = load_model("model.h5")

# Load the tokenizer
with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

# Load the configuration
with open("config.pkl", "rb") as f:
    config = pickle.load(f)

max_len = config["max_len"]
vocab_size = config["vocab_size"]

print(f"Model loaded.")
print(f"Vocabulary size: {vocab_size}")
print(f"Max sequence length: {max_len}")


Model loaded.
Vocabulary size: 421
Max sequence length: 14


#### **3.2 Build the Index-to-Word Mapping**
**Instructions**: The tokenizer stores `word -> index`. To convert model predictions (integers) back into readable words, we need the reverse mapping `index -> word`.


In [4]:
# Reverse the tokenizer dictionary to get index -> word
index_to_word = {v: k for k, v in tokenizer.word_index.items()}

# Look at the first 10 entries
print("Index to Word Sample:", dict(list(index_to_word.items())[:10]))

Index to Word Sample: {1: 'is', 2: 'the', 3: 'in', 4: 'it', 5: 'you', 6: 'what', 7: 'to', 8: 'a', 9: 'for', 10: 'where'}


### **4. Prediction Function**
#### **4.1 Define `predict_answer`**
**Instructions**: Write a function that takes a string input from the user, tokenizes it, pads it, runs it through the model, and converts the predicted integers back into words.


In [5]:
def predict_answer(text):
    # Step 1: Convert input text to lowercase and tokenize
    seq = tokenizer.texts_to_sequences([text.lower()])
    
    # Step 2: Pad the sequence to max_len
    seq = pad_sequences(seq, maxlen=max_len, padding='post')
    
    # Step 3: Get prediction from the model
    pred = model.predict(seq, verbose=0)
    
    # Step 4: Take the most likely word at each position (greedy decoding)
    pred_ids = np.argmax(pred, axis=-1)[0]
    
    # Step 5: Convert integers back to words, skip padding (0)
    words = []
    for idx in pred_ids:
        if idx == 0:
            continue
        word = index_to_word.get(idx, '')
        if word:
            words.append(word)
    
    return " ".join(words) if words else "(no response)"

#### Explanation
The function follows the same pipeline that was used during training, only in reverse for the final step.

**1. Tokenize the input**
The user enters a sentence in English. We lowercase it and convert each word into an integer using the same `tokenizer` that was fit on the training data. Words that the tokenizer has never seen will not appear in `word_index` and will simply be dropped.

**2. Pad the sequence**
Just like during training, every input must have the same length (`max_len`). We pad with zeros at the end (`padding='post'`).

**3. Run the model**
The model output is an array of shape `(1, max_len, vocab_size)`. At each position, the model gives a probability for every word in the vocabulary.

**4. Greedy decoding (argmax)**
At each position, we pick the **single word with the highest probability**. This is the simplest decoding method. More advanced methods like beam search or sampling could produce more diverse responses but greedy is enough to demonstrate the model's behavior.

**5. Convert back to words**
We use `index_to_word` to turn the integers back into words, skipping padding tokens (0).

### **5. Testing the Chatbot**
#### **5.1 In-Domain Questions**
**Instructions**: Test the chatbot with questions that are similar in style to the training data. This gives us a baseline for what "working" looks like.


In [6]:
in_domain_questions = [
    "Where is Angkor Wat?",
    "What is the capital of Cambodia?",
    "What food should I try in Cambodia?",
    "What currency is used in Cambodia?",
    "When is the best time to visit Cambodia?",
]

for q in in_domain_questions:
    print(f"Q: {q}")
    print(f"A: {predict_answer(q)}")
    print("-" * 60)

Q: Where is Angkor Wat?
A: it is is in in in in in in in in in in in
------------------------------------------------------------
Q: What is the capital of Cambodia?
A: it is gmt plus the are are are are are are are are are
------------------------------------------------------------
Q: What food should I try in Cambodia?
A: it should try amok and lok lak lak lak lak lak lak lak lak
------------------------------------------------------------
Q: What currency is used in Cambodia?
A: it is and and and widely widely widely widely widely widely widely widely widely
------------------------------------------------------------
Q: When is the best time to visit Cambodia?
A: it is is is the best time
------------------------------------------------------------


#### **5.2 Out-of-Vocabulary (OOV) Words**
**Instructions**: Test the chatbot with questions that contain words it has never seen during training. The tokenizer cannot convert these words to integers, so the model has very little to work with.


In [7]:
oov_questions = [
    "Tell me about cryptocurrency in Cambodia",
    "What about quantum computing tourism?",
    "Can I find vegan keto restaurants?",
]

for q in oov_questions:
    print(f"Q: {q}")
    print(f"A: {predict_answer(q)}")
    print("-" * 60)

Q: Tell me about cryptocurrency in Cambodia
A: it is is and for for for for for for for for for for
------------------------------------------------------------
Q: What about quantum computing tourism?
A: it is is is is is is is is is is is is is
------------------------------------------------------------
Q: Can I find vegan keto restaurants?
A: it should khan khan khan khan khan khan khan khan khan khan khan khan
------------------------------------------------------------


#### **5.3 Paraphrasing**
**Instructions**: Ask the same underlying question in different surface forms. A robust language model should give similar answers for paraphrases.


In [8]:
paraphrase_questions = [
    "What food can I eat in Cambodia?",
    "What dishes are popular in Cambodia?",
    "Tell me about Cambodian cuisine",
    "What should I eat when visiting Cambodia?",
]

for q in paraphrase_questions:
    print(f"Q: {q}")
    print(f"A: {predict_answer(q)}")
    print("-" * 60)

Q: What food can I eat in Cambodia?
A: it should cover your knees or or or or or or or or or
------------------------------------------------------------
Q: What dishes are popular in Cambodia?
A: it it is is and and and and and and and and and and
------------------------------------------------------------
Q: Tell me about Cambodian cuisine
A: it is is is is is is is is is is is is is
------------------------------------------------------------
Q: What should I eat when visiting Cambodia?
A: it temple a temple and and and and and and and and and and
------------------------------------------------------------


#### **5.4 Long Inputs**
**Instructions**: SimpleRNN suffers from the vanishing gradient problem, which means information from the beginning of a long sequence is effectively lost by the time the network reaches the end. We test this by feeding very long questions.


In [9]:
long_questions = [
    "I am planning a trip to Cambodia next month with my family and we have never been to Southeast Asia before, so could you tell me what the most important things to know about Cambodian culture are?",
    "My friends and I are backpackers visiting many countries in Asia and we want to spend about two weeks in Cambodia, where should we go?",
]

for q in long_questions:
    print(f"Q: {q}")
    print(f"A: {predict_answer(q)}")
    print("-" * 60)

Q: I am planning a trip to Cambodia next month with my family and we have never been to Southeast Asia before, so could you tell me what the most important things to know about Cambodian culture are?
A: it it is is for ferry the sihanoukville
------------------------------------------------------------
Q: My friends and I are backpackers visiting many countries in Asia and we want to spend about two weeks in Cambodia, where should we go?
A: it it is is and tuk tuks from filtered water
------------------------------------------------------------


#### **5.5 Short Inputs**
**Instructions**: At the opposite extreme, very short inputs give the model almost no context. Most of the padded sequence will be zeros, leaving the model with very little signal to condition on.


In [10]:
short_questions = [
    "food",
    "hello",
    "Cambodia?",
]

for q in short_questions:
    print(f"Q: {q}")
    print(f"A: {predict_answer(q)}")
    print("-" * 60)

Q: food
A: blue blue blue blue blue blue blue blue blue blue blue blue blue blue
------------------------------------------------------------
Q: hello
A: amok amok amok amok amok amok amok amok amok amok amok amok amok amok
------------------------------------------------------------
Q: Cambodia?
A: used used used used used used used used used used used used used used
------------------------------------------------------------


#### **5.6 Off-Topic Questions**
**Instructions**: The training data is restricted to Cambodia tourism. When asked about unrelated topics, the model has no relevant learned patterns. We expect generic, repetitive, or topically incorrect responses.


In [11]:
off_topic_questions = [
    "What is the meaning of life?",
    "How do I bake a chocolate cake?",
    "Who won the World Cup last year?",
]

for q in off_topic_questions:
    print(f"Q: {q}")
    print(f"A: {predict_answer(q)}")
    print("-" * 60)

Q: What is the meaning of life?
A: it is gmt 60km 60km 60km 60km 60km 60km 60km 60km 60km 60km 60km
------------------------------------------------------------
Q: How do I bake a chocolate cake?
A: passapp can take a a a a a a a a a a a
------------------------------------------------------------
Q: Who won the World Cup last year?
A: drink racing cheap 15 15 15 15 15 15 15 15 15 15 15
------------------------------------------------------------


### **6. Summary Table**
**Instructions**: Collect all the test results into a single DataFrame so we can compare the behavior across categories side by side.


In [12]:
test_set = [
    ("In-domain",   "Where is Angkor Wat?"),
    ("In-domain",   "What is the capital of Cambodia?"),
    ("In-domain",   "What currency is used in Cambodia?"),
    ("OOV words",   "Tell me about cryptocurrency in Cambodia"),
    ("Paraphrase",  "What dishes are popular in Cambodia?"),
    ("Paraphrase",  "Tell me about Cambodian cuisine"),
    ("Long input",  "I am planning a trip to Cambodia next month with my family and we have never been to Southeast Asia before, so could you tell me what the most important things to know about Cambodian culture are?"),
    ("Short input", "food"),
    ("Off-topic",   "What is the meaning of life?"),
    ("Off-topic",   "How do I bake a chocolate cake?"),
]

rows = []
for category, q in test_set:
    a = predict_answer(q)
    rows.append({"Category": category, "Question": q, "Response": a})

results_df = pd.DataFrame(rows)
results_df

,Category,Question,Response
0,In-domain,Where is Angkor Wat?,it is is in in in in in in in in in in in
1,In-domain,What is the capital of Cambodia?,it is gmt plus the are are are are are are are...
2,In-domain,What currency is used in Cambodia?,it is and and and widely widely widely widely ...
3,OOV words,Tell me about cryptocurrency in Cambodia,it is is and for for for for for for for for f...
4,Paraphrase,What dishes are popular in Cambodia?,it it is is and and and and and and and and an...
5,Paraphrase,Tell me about Cambodian cuisine,it is is is is is is is is is is is is is
6,Long input,I am planning a trip to Cambodia next month wi...,it it is is for ferry the sihanoukville
7,Short input,food,blue blue blue blue blue blue blue blue blue b...
8,Off-topic,What is the meaning of life?,it is gmt 60km 60km 60km 60km 60km 60km 60km 6...
9,Off-topic,How do I bake a chocolate cake?,passapp can take a a a a a a a a a a a


### **7. Error Analysis**
#### **7.1 Failure Modes Observed**



**1. Repetition Collapse**

The model often gets stuck repeating the same word (for example `"in in in in"` or `"lak lak lak"`). Two reasons combine to cause this:

- The prediction function uses greedy decoding (`np.argmax`), which picks the single most probable word at every position with no diversity.
- SimpleRNN has the vanishing gradient problem. It cannot remember many steps back. So after predicting "lak", by the next step it has already "forgotten" that it just said "lak", and "lak" remains the highest-probability word.

**2. Insensitivity to Paraphrasing**

Different surface forms of the same question often produce different responses. The model relies on shallow word co-occurrence rather than semantic understanding. There is no mechanism for synonym handling.

**3. Loss of Context on Long Inputs**

Long questions of 30+ words produce extremely short responses (for example, just `"of the pepper"`). This is the vanishing gradient effect — the contribution of early tokens decays exponentially with sequence length. The model only conditions on the last few words of long inputs.

**4. Degenerate Behavior on Short Inputs**

When the input is just one or two words (for example `"food"`), most positions in the padded sequence are zeros. The model has very little signal to condition on, and the response collapses to generic high-frequency tokens.

**5. No Topic Boundary**

The model has no concept of "I don't know" or "this is off-topic". When asked an unrelated question (`"How do I bake a chocolate cake?"`), it still answers with Cambodia tourism words (`"passapp can take a visa night"`). This is because the only vocabulary it has learned is Cambodia tourism.

**6. Out-of-Vocabulary Words**

Unknown words (`"cryptocurrency"`, `"quantum"`) are dropped by the tokenizer because they were never seen during training. The input loses all distinguishing information for those words, so the response cannot meaningfully depend on what the user actually wrote.

**7. Misleading Accuracy Metric**

Despite the model reaching around 59% validation accuracy during training, the qualitative responses are poor. This is because token-level accuracy on padded sequences is inflated by correctly predicted padding tokens and common words like `"is"`, `"the"`, `"and"`, while the content words remain wrong.

### **8. Conclusion**

These failure modes are not bugs in the implementation. They are well-known limitations of the SimpleRNN architecture, which is exactly why later architectures such as **LSTM**, **GRU**, and most importantly **Transformer-based** models with attention were developed.

To improve this chatbot, the recommended next steps would be:

- Replace SimpleRNN with **LSTM** or **GRU** to handle long-term dependencies.
- Add **Dropout** to reduce overfitting.
- Use **early stopping** based on validation loss to avoid training past the best epoch.
- Use **beam search** or **sampling** instead of greedy decoding to reduce repetition.
- Use a **Transformer with attention** for the strongest baseline.
- Increase the dataset size beyond 2000 question-answer pairs.